# Full run — GeoWeighting A1 + cached CFM mechanism

This notebook runs both next-stage experiments in one Kaggle session. It overlaps the inexpensive cached-feature CFM mechanism test with the final GeoWeighting seed so both T4s remain useful. The baseline and specialized models are not rerun.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2 for the intended runtime'
print('Detected GPUs:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## Securely clone the current repository

Create a Kaggle secret named `github_token`. The token is passed through a temporary askpass helper and is never embedded in the clone URL or notebook output.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing or empty'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
assert (PROJECT_ROOT / 'scripts' / 'run_combined_interventions.py').is_file(), 'Commit and push the combined-run files first'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Validate the uploaded baseline and lock both protocols

In [ ]:
from datetime import datetime, timezone
import numpy as np, pandas as pd, yaml
from baseline_artifacts import find_confirmatory_root, compute_anchor_geometry_prior
from cfm_mechanism import load_feature_bank, fixed_sample_split
INPUT_ROOT = Path('/kaggle/input/datasets/dyhngg/checkpoint-new-prune')
BASELINE_ROOT = find_confirmatory_root(INPUT_ROOT)
A1_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_geoweighting_a1.yaml'
CFM_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_cfm_mechanism.yaml'
a1_config = yaml.safe_load(A1_CONFIG.read_text())
cfm_config = yaml.safe_load(CFM_CONFIG.read_text())
assert a1_config['experiment']['seeds'] == [0, 1, 2] and a1_config['training']['epochs'] == 20
assert a1_config['compression']['train_widths'] == [0.25, 0.50, 0.75, 1.00]
assert a1_config['geometry_weighting'] == {'alpha': 1.0, 'beta': 0.5, 'epsilon': 1e-8}
assert cfm_config['cfm']['tasks'] == [{'known': [0.30, 0.60, 0.80], 'holdout': 0.40}, {'known': [0.30, 0.40, 0.80], 'holdout': 0.60}]
prior, anchor_weights = compute_anchor_geometry_prior(BASELINE_ROOT, a1_config['compression']['train_widths'], **a1_config['geometry_weighting'])
assert np.isclose(np.mean(list(anchor_weights.values())), 1.0) and all(value > 0 for value in anchor_weights.values())
reference_ids = None
for seed in [0, 1, 2]:
    bank, sample_ids = load_feature_bank(BASELINE_ROOT, seed, [0.30, 0.40, 0.60, 0.80])
    assert len(sample_ids) == 2000 and all(tensor.shape == (2000, 128) for tensor in bank.values())
    if reference_ids is None:
        reference_ids = sample_ids
    else:
        assert torch.equal(reference_ids, sample_ids), 'Feature sample IDs differ across seeds'
train_idx, validation_idx, test_idx = fixed_sample_split(2000, cfm_config['cfm']['sample_split_seed'], 0.70, 0.15)
assert (len(train_idx), len(validation_idx), len(test_idx)) == (1400, 300, 300)
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-a1-cfm-full-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
print('Resolved baseline:', BASELINE_ROOT)
display(prior)
print('Frozen A1 weights:', anchor_weights)
print('Cached-feature and split checks: OK')

## Run both pipelines

The two execution lanes are `GPU 0: A1 seed 0 -> A1 seed 2` and `GPU 1: A1 seed 1 -> CFM`. CFM depends only on cached baseline representations, so it safely overlaps the final A1 seed and does not retrain the backbone. Expected total runtime is roughly 1.5–2.3 hours.

In [ ]:
from scripts.run_combined_interventions import run_combined
started = time.perf_counter()
result = run_combined(A1_CONFIG, CFM_CONFIG, RUN_DIR, gpu_ids=[0, 1])
print(f"Full run completed in {(time.perf_counter() - started) / 3600:.2f} hours")
print('Combined report:', result['report'])

## Independent decisions and combined verdict

In [ ]:
import json
from IPython.display import Markdown, display
decision = json.loads((RUN_DIR / 'combined_decision.json').read_text())
display(Markdown(f"## {decision['combined_verdict']}"))
print('Runtime:', result['timings'])
print('A1 checks:', result['a1_checks'])
display(result['a1_comparison'])
print('CFM checks:', result['cfm_checks'])
display(result['cfm_comparison'])
display(Markdown((RUN_DIR / 'combined_report.md').read_text()))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for label, frame in [('Baseline', result['baseline']), ('GeoWeighting A1', result['a1'])]:
    accuracy = frame.groupby('budget')['accuracy'].mean()
    axes[0].plot(accuracy.index, accuracy, marker='o', label=label)
    geometry = frame.dropna(subset=['local_wasserstein_sensitivity']).groupby('budget')['local_wasserstein_sensitivity'].mean()
    axes[1].plot(geometry.index, geometry, marker='o', label=label)
cfm_plot = result['cfm_summary'].sort_values(['holdout_width', 'sliced_wasserstein'])
methods = list(cfm_plot['method'].unique())
holdouts = sorted(cfm_plot['holdout_width'].unique())
x = np.arange(len(holdouts)); bar_width = 0.18
for index, method in enumerate(methods):
    values = [cfm_plot.loc[(cfm_plot['holdout_width'] == width) & (cfm_plot['method'] == method), 'sliced_wasserstein'].iloc[0] for width in holdouts]
    axes[2].bar(x + (index - 1.5) * bar_width, values, bar_width, label=method)
axes[0].set(xlabel='Width', ylabel='Mean test accuracy', title='Dense-budget accuracy')
axes[1].set(xlabel='Interval start c', ylabel='Mean G(c)', title='Local geometry')
axes[2].set(xlabel='Held-out width', ylabel='Mean sliced Wasserstein', title='CFM mechanism')
axes[2].set_xticks(x, [f'{width:.2f}' for width in holdouts])
for axis in axes:
    axis.grid(alpha=.25)
    axis.legend(fontsize=8)
fig.tight_layout()
plot_path = RUN_DIR / 'combined_summary.png'
fig.savefig(plot_path, dpi=180)
plt.show()

## Validate and download the complete artifact

In [ ]:
import shutil, zipfile
from IPython.display import FileLink
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())
manifest = pd.DataFrame({'relative_path': [str(path.relative_to(RUN_DIR)) for path in files], 'size_bytes': [path.stat().st_size for path in files]})
manifest.to_csv(RUN_DIR / 'artifact_manifest.csv', index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
with zipfile.ZipFile(archive) as bundle:
    names = set(bundle.namelist())
required = [
    'combined_report.md', 'combined_decision.json', 'combined_summary.png',
    'geoweighting_a1/frozen_anchor_geometry_prior.csv',
    'geoweighting_a1/baseline_a1_paired_comparison.csv',
    'geoweighting_a1/seed_0/checkpoint.pt',
    'geoweighting_a1/seed_1/checkpoint.pt',
    'geoweighting_a1/seed_2/checkpoint.pt',
    'cfm_mechanism/cfm_mechanism_results.csv',
    'cfm_mechanism/cfm_paired_comparison.csv',
    'cfm_mechanism/cfm_method_summary.csv',
]
missing = [name for name in required if name not in names]
assert not missing, f'Missing required artifacts: {missing}'
print('Validated artifact count:', len(names))
print('Archive:', archive)
display(FileLink(str(archive)))
print('Use Save Version after this cell completes so Kaggle persists the ZIP.')